In [ ]:
import os

os.environ["KERAS_BACKEND"] = "torch"
os.environ["CUDA_VISIBLE_DEVICES"] = "1"

import keras

keras.config.set_image_data_format("channels_first")

import torch
import numpy as np
import albumentations as A

from pathlib import Path
from typing import Sequence

from PIL import Image
from torch.utils.data import Dataset, DataLoader
from albumentations.pytorch import ToTensorV2


class UnlabeledImageDataset(Dataset):
    def __init__(self, root_dir: Path, cond_shape: Sequence[int], transform=None):
        self.root_dir = root_dir
        self.transform = transform
        self.cond_shape = cond_shape
        # Get list of all image file names in the folder
        self.image_files = list(root_dir.glob("*.bmp"))

    def __len__(self):
        return len(self.image_files)

    def __getitem__(self, idx):
        img_name = self.image_files[idx]
        image = Image.open(img_name).convert("L")

        if self.transform:
            image = self.transform(image=np.array(image))
            image = image["image"]

        condition = np.ones(self.cond_shape, dtype=np.float32)

        device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

        image = image.to(device)
        return (image, torch.tensor(condition, device=device)), image


def train_transforms(img_size, mean=[0.7], std=[0.4]):
    return A.Compose(
        [
            A.InvertImg(1),
            A.Resize(img_size, img_size),
            A.HorizontalFlip(0.5),
            A.Affine(scale=(0.9, 0.95), rotate=(15, 90), shear=(-2, 2), p=0.5),
            A.RandomRotate90(0.5),
            A.RandomBrightnessContrast(
                brightness_limit=0.2, contrast_limit=(-0.5, 0.5), p=0.5
            ),
            A.GaussianBlur(blur_limit=(1, 3), p=0.3),
            A.Normalize(mean=mean, std=std),
            ToTensorV2(),
        ]
    )


def valid_transforms(img_size, mean=[0.7], std=[0.4]):
    return A.Compose(
        [
            A.InvertImg(1),
            A.Resize(img_size, img_size),
            A.Normalize(mean=mean, std=std),
            ToTensorV2(),
        ]
    )

In [ ]:
import keras
import torch
from torchvision.models import mobilenet_v3_small

keras.backend.clear_session()

model = mobilenet_v3_small(weights="DEFAULT")
feature_layers = list(model.get_submodule("features").children())

# x = keras.random.normal((1, 3, 224, 224)).cpu()
# for idx, layer in enumerate(feature_layers):
#     x = layer(x)
#     print(idx, x.shape)

# 0 torch.Size([1, 16, 112, 112])
# 1 torch.Size([1, 16, 56, 56])
# 2 torch.Size([1, 24, 28, 28])
# 3 torch.Size([1, 24, 28, 28])
# 4 torch.Size([1, 40, 14, 14])
# 5 torch.Size([1, 40, 14, 14])
# 6 torch.Size([1, 40, 14, 14])
# 7 torch.Size([1, 48, 14, 14])
# 8 torch.Size([1, 48, 14, 14])
# 9 torch.Size([1, 96, 7, 7])
# 10 torch.Size([1, 96, 7, 7])
# 11 torch.Size([1, 96, 7, 7])
# 12 torch.Size([1, 576, 7, 7])


tap_indices = [0, 1, 2, 4, 9, 14]
grouped = [feature_layers[i:j] for i, j in zip(tap_indices, tap_indices[1:])]

backbone = []
for group in grouped:
    if len(group) == 1:
        backbone.append(keras.layers.TorchModuleWrapper(group[0]))
    else:
        backbone.append(keras.layers.TorchModuleWrapper(torch.nn.Sequential(*group)))

# x = keras.random.normal((1, 3, 224, 224)).cuda()
# for idx, layer in enumerate(backbone):
#     x = layer(x)
#     print(idx, x.shape)

# 0 torch.Size([1, 16, 112, 112])
# 1 torch.Size([1, 16, 56, 56])
# 2 torch.Size([1, 24, 28, 28])
# 3 torch.Size([1, 48, 14, 14])
# 4 torch.Size([1, 576, 7, 7])

In [ ]:
from agx_core.models.ra.encoder import Encoder
from agx_core.models.ra.decoder import Decoder
from agx_torch.models.ra.model import ReversedAutoencoder

enc = Encoder(backbone, latent_size=512)

img_size = 224
res = img_size // 2 ** len(enc.backbone)

img_shape = (None, 1, img_size, img_size)
cond_shape = (None, 1, res, res)
print(img_shape, cond_shape)

dec = Decoder(target_shape=img_shape[1:])

In [ ]:
train_path = Path("../data/products/LaTuaPastaGlassJars/Clean/train/")
valid_path = Path("../data/products/LaTuaPastaGlassJars/Clean/val/")


ds_train = UnlabeledImageDataset(
    train_path, transform=train_transforms(img_size), cond_shape=cond_shape[1:]
)
ds_valid = UnlabeledImageDataset(
    valid_path, transform=valid_transforms(img_size), cond_shape=cond_shape[1:]
)

loader_train = DataLoader(ds_train, batch_size=8, shuffle=True)
loader_valid = DataLoader(ds_valid, batch_size=8)

In [ ]:
# import matplotlib.pyplot as plt

# fig, axes = plt.subplots(3, 3, figsize=(12, 12))

# for idx, ax in enumerate(axes.flat):
#     X, y = ds_train[idx]
#     ax.imshow(X[0][0], cmap="gray")
#     ax.set_title(f"Image {idx + 1}")
#     ax.axis("off")

# plt.tight_layout()
# plt.show()

In [ ]:
from keras.optimizers import Adam
from keras.callbacks import ModelCheckpoint

from agx_core.models.ra.optimizer import RAOptimizer
from agx_core.models.ra.callbacks import (
    AdversarialEquilibriumCallback,
    BackboneThawCallback,
)

ra = ReversedAutoencoder(enc, dec, beta_kld=0.1)
ra.build([img_shape, cond_shape])

optimizer = RAOptimizer(Adam(learning_rate=1e-6), Adam(learning_rate=1e-4))
ra.compile(optimizer)

ra.save("ra.test.keras")

callbacks = [
    AdversarialEquilibriumCallback(0.0, -1),
    BackboneThawCallback(),
    ModelCheckpoint(
        filepath="ra_mbnetv3.model.keras",
        monitor="val_loss_rec",
        mode="min",
        save_best_only=True,
        initial_value_threshold=0.16984,
        verbose=1,
    ),
]

ra_test = keras.models.load_model("ra.test.keras", safe_mode=False)

In [ ]:
history = ra_test.fit(loader_train, validation_data=loader_valid, epochs=400, initial_epoch=200, callbacks=callbacks, verbose=2)

In [ ]:
import pandas as pd

df = pd.DataFrame.from_dict(history.history)

hist_file = Path("history.csv")
if hist_file.exists():
    hist = pd.read_csv(hist_file)
    df.index += len(hist)
    hist = pd.concat([hist, df])
hist.tail()


In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(3, 3, figsize=(12, 12))

for idx, ax in enumerate(axes.flat):
    (I, C), y = ds_valid[idx]
    rec = ra([I[np.newaxis, ...], C[np.newaxis, ...]])
    ax.imshow(rec.cpu().detach().numpy()[0][0], cmap="gray")
    ax.set_title(f"Image {idx + 1}")
    ax.axis("off")

plt.tight_layout()
plt.show()

In [ ]:
import pandas as pd

fig, ax = plt.subplots(figsize=(21, 14))

# Adjust val_loss_embed, forgot to update validation scaling
hist[["loss_enc", "val_loss_enc"]].plot(ax=ax)

In [ ]:
# device = torch.device("cuda")
# ra.eval()
# ra = ra.to(device)
# I, C = torch.rand((1, *img_shape[1:]), device=device), torch.rand(
#     (1, *cond_shape[1:]), device=device
# )

# torch.onnx.export(
#     ra.encoder,
#     ([I, C],),
#     "model.onnx",
#     input_names=["image", "condition"],
#     output_names=["reconstruction"],
# )